#  SocialSentinel-AI: Cresci-2017 Exploratory Data Analysis (EDA)

**Objective**: Inspect, validate, clean, and profile the **Cresci-2017** benchmark dataset for automated social bot detection.

### Dataset Structure Overview:
- **Genuine Accounts** (`genuine_accounts`): Verified human accounts (~3,474 users).
- **Social Spambots** (`social_spambots_1`, `social_spambots_2`, `social_spambots_3`): Advanced modern bots displaying coordinated behavior.
- **Traditional Spambots** (`traditional_spambots_1`): Simple spam accounts with spam tweets.
- **Fake Followers** (`fake_followers`): Purchased low-activity accounts designed to inflate follower counts.

## 1. Imports and Setup

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Project root directory
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = ROOT_DIR / 'data' / 'raw' / 'cresci2017'
INTERIM_DATA_DIR = ROOT_DIR / 'data' / 'interim'

print(f"Project root: {ROOT_DIR}")
print(f"Raw data directory: {RAW_DATA_DIR}")

: 

## 2. Cresci-2017 Data Loader Function
The Cresci-2017 dataset typically unpacks into subdirectories containing `users.csv` and `tweets.csv`.
Here we create a robust loader to parse user profiles and assign ground-truth binary and multiclass labels.

In [ ]:
def load_cresci_users(base_path: Path) -> pd.DataFrame:
    """
    Scans the cresci2017 directory and loads user profiles from sub-folders or direct CSV files.
    Categorizes users into binary (0=Human, 1=Bot) and specific account categories.
    """
    categories = {
        'genuine_accounts': {'label': 0, 'category': 'genuine'},
        'social_spambots_1': {'label': 1, 'category': 'social_spambot_1'},
        'social_spambots_2': {'label': 1, 'category': 'social_spambot_2'},
        'social_spambots_3': {'label': 1, 'category': 'social_spambot_3'},
        'traditional_spambots_1': {'label': 1, 'category': 'traditional_spambot'},
        'fake_followers': {'label': 1, 'category': 'fake_followers'}
    }
    
    dfs = []
    
    # Check if folder exists
    if not base_path.exists():
        print(f"⚠️ Directory {base_path} not found. Please extract your Cresci-2017 files here.")
        return pd.DataFrame()
        
    # Find all users.csv or corresponding CSV files
    found_files = list(base_path.glob('**/users.csv')) + list(base_path.glob('*.csv'))
    
    if not found_files:
        print(f"⚠️ No user CSV files found under {base_path}.")
        print("Expected layout example: data/raw/cresci2017/genuine_accounts/users.csv")
        return pd.DataFrame()
        
    for filepath in found_files:
        # Determine category based on folder or file name
        cat_key = None
        for key in categories:
            if key in str(filepath).lower():
                cat_key = key
                break
                
        try:
            df = pd.read_csv(filepath, low_memory=False, encoding='latin1')
            if cat_key:
                df['is_bot'] = categories[cat_key]['label']
                df['dataset_category'] = categories[cat_key]['category']
            else:
                df['is_bot'] = np.nan
                df['dataset_category'] = filepath.stem
                
            df['source_file'] = filepath.name
            dfs.append(df)
            print(f"✅ Loaded {len(df):,} records from {filepath.relative_to(base_path)}")
        except Exception as e:
            print(f"❌ Failed reading {filepath}: {e}")
            
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"\nTotal accounts loaded: {len(combined_df):,}")
        return combined_df
    return pd.DataFrame()

# Load dataset
users_df = load_cresci_users(RAW_DATA_DIR)

## 3. High-Level Dataset Inspection & Summary

In [ ]:
if not users_df.empty:
    print("=== Head (First 5 records) ===")
    display(users_df.head())
    
    print("\n=== Info & Types ===")
    users_df.info()
else:
    print("Dataset is empty. Place Cresci-2017 files in data/raw/cresci2017/ and re-run cell 2.")

## 4. Missing Values Audit

In [ ]:
if not users_df.empty:
    null_pct = (users_df.isnull().sum() / len(users_df) * 100).sort_values(ascending=False)
    null_summary = pd.DataFrame({'Missing Count': users_df.isnull().sum(), 'Missing %': null_pct})
    print("=== Missing Values Summary ===")
    display(null_summary[null_summary['Missing Count'] > 0])
    
    # Visualize top missing columns
    plt.figure(figsize=(10, 4))
    null_pct.head(15).plot(kind='bar', color='coral')
    plt.title('Top 15 Features with Missing Values (%)')
    plt.ylabel('Percentage Missing (%)')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 5. Target Class Distribution (Bots vs. Humans)

In [ ]:
if not users_df.empty and 'is_bot' in users_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Binary target counts
    users_df['is_bot'].value_counts().rename({0: 'Human (0)', 1: 'Bot (1)'}).plot(
        kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], rot=0
    )
    axes[0].set_title('Binary Label Distribution (Human vs. Bot)')
    axes[0].set_ylabel('Account Count')
    
    # 2. Detailed Category breakdown
    if 'dataset_category' in users_df.columns:
        users_df['dataset_category'].value_counts().plot(
            kind='barh', ax=axes[1], color='#3498db'
        )
        axes[1].set_title('Account Breakdown by Dataset Category')
        axes[1].set_xlabel('Account Count')
        
    plt.tight_layout()
    plt.show()

## 6. Key Numerical Behavioral Feature Analysis
Key signals in Twitter bot classification:
- `followers_count`
- `friends_count` (following)
- `statuses_count` (tweet count)
- `favourites_count`
- `reputation_ratio`: `followers / (followers + friends)`

In [ ]:
if not users_df.empty:
    numeric_cols = ['followers_count', 'friends_count', 'statuses_count', 'favourites_count']
    valid_numeric = [c for c in numeric_cols if c in users_df.columns]
    
    for col in valid_numeric:
        users_df[col] = pd.to_numeric(users_df[col], errors='coerce').fillna(0)
        
    # Compute reputation ratio
    if 'followers_count' in users_df.columns and 'friends_count' in users_df.columns:
        users_df['reputation_ratio'] = users_df['followers_count'] / (
            users_df['followers_count'] + users_df['friends_count'] + 1e-5
        )
        valid_numeric.append('reputation_ratio')
        
    print("=== Summary Statistics by Label ===")
    display(users_df.groupby('is_bot')[valid_numeric].describe().T)
    
    # Distribution plots (Log-scale)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    if 'followers_count' in users_df.columns:
        sns.boxplot(data=users_df, x='is_bot', y=np.log1p(users_df['followers_count']), ax=axes[0], palette=['#2ecc71', '#e74c3c'])
        axes[0].set_xticklabels(['Human', 'Bot'])
        axes[0].set_title('Log(Followers Count + 1) by Class')
        
    if 'reputation_ratio' in users_df.columns:
        sns.kdeplot(data=users_df, x='reputation_ratio', hue='is_bot', common_norm=False, fill=True, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
        axes[1].set_title('Reputation Ratio Density [Followers / (Followers + Friends)]')
        
    plt.tight_layout()
    plt.show()

## 7. Next Steps & Feature Engineering Plan
- [ ] Compute account age from `created_at`
- [ ] Extract textual metadata features from `description` and `name` (entropy, digits count, length)
- [ ] Export clean interim merged dataset to `data/interim/cresci2017_users_clean.csv`
- [ ] Build baseline classification models in `src/models/`